# Thử thách: Phân tích văn bản về Khoa học Dữ liệu

Trong ví dụ này, hãy làm một bài tập đơn giản bao gồm tất cả các bước của quy trình khoa học dữ liệu truyền thống. Bạn không cần phải viết mã, chỉ cần nhấp vào các ô bên dưới để chạy chúng và quan sát kết quả. Như một thử thách, bạn được khuyến khích thử mã này với dữ liệu khác nhau.

## Mục tiêu

Trong bài học này, chúng ta đã thảo luận về các khái niệm khác nhau liên quan đến Khoa học Dữ liệu. Hãy thử khám phá thêm các khái niệm liên quan bằng cách làm một số **khai thác văn bản**. Chúng ta sẽ bắt đầu với một văn bản về Khoa học Dữ liệu, trích xuất các từ khóa từ đó, và sau đó thử trực quan hóa kết quả.

Là một văn bản, tôi sẽ sử dụng trang về Khoa học Dữ liệu từ Wikipedia:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Bước 1: Lấy dữ liệu

Bước đầu tiên trong mọi quy trình khoa học dữ liệu là lấy dữ liệu. Chúng ta sẽ sử dụng thư viện `requests` để làm điều đó:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Bước 2: Biến đổi Dữ liệu

Bước tiếp theo là chuyển đổi dữ liệu thành dạng phù hợp để xử lý. Trong trường hợp của chúng ta, chúng ta đã tải xuống mã nguồn HTML từ trang, và chúng ta cần chuyển nó thành văn bản thuần.

Có nhiều cách để làm điều này. Chúng ta sẽ sử dụng [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), một thư viện Python phổ biến để phân tích cú pháp HTML. BeautifulSoup cho phép chúng ta nhắm mục tiêu các phần tử HTML cụ thể, vì vậy chúng ta có thể tập trung vào nội dung bài viết chính từ Wikipedia và giảm bớt một số menu điều hướng, thanh bên, chân trang và các nội dung không liên quan khác (mặc dù một số đoạn văn bản mẫu có thể vẫn còn).


Trước tiên, chúng ta cần cài đặt thư viện BeautifulSoup để phân tích HTML:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Bước 3: Thu thập Thông tin

Bước quan trọng nhất là biến dữ liệu của chúng ta thành một dạng mà từ đó chúng ta có thể rút ra những hiểu biết. Trong trường hợp của chúng ta, chúng ta muốn trích xuất các từ khóa từ văn bản, và xem từ khóa nào có ý nghĩa hơn.

Chúng ta sẽ sử dụng thư viện Python gọi là [RAKE](https://github.com/aneesha/RAKE) để trích xuất từ khóa. Trước tiên, hãy cài đặt thư viện này trong trường hợp nó chưa có: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

Chức năng chính có sẵn từ đối tượng `Rake`, mà chúng ta có thể tùy chỉnh bằng một số tham số. Trong trường hợp của chúng ta, chúng ta sẽ đặt độ dài tối thiểu của một từ khóa là 5 ký tự, tần suất tối thiểu của một từ khóa trong tài liệu là 3, và số từ tối đa trong một từ khóa là 2. Hãy thoải mái thử nghiệm với các giá trị khác và quan sát kết quả.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Chúng tôi đã thu được một danh sách các thuật ngữ cùng với mức độ quan trọng liên quan. Như bạn có thể thấy, các lĩnh vực có liên quan nhất, chẳng hạn như học máy và dữ liệu lớn, xuất hiện trong danh sách ở các vị trí hàng đầu.

## Bước 4: Trực quan hóa Kết quả

Mọi người có thể hiểu dữ liệu tốt nhất ở dạng trực quan. Do đó thường có ý nghĩa khi trực quan hóa dữ liệu để rút ra một số hiểu biết. Chúng ta có thể sử dụng thư viện `matplotlib` trong Python để vẽ phân phối đơn giản của các từ khóa cùng mức độ liên quan của chúng:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Tuy nhiên, có một cách còn tốt hơn để trực quan hóa tần suất từ - sử dụng **Word Cloud**. Chúng ta sẽ cần cài đặt thêm một thư viện nữa để vẽ word cloud từ danh sách từ khóa của chúng ta.


In [ ]:
!{sys.executable} -m pip install wordcloud

Đối tượng `WordCloud` chịu trách nhiệm nhận vào hoặc văn bản gốc, hoặc danh sách các từ đã được tính trước cùng tần suất của chúng, và trả về một hình ảnh, sau đó có thể được hiển thị bằng `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Chúng ta cũng có thể truyền vào văn bản gốc cho `WordCloud` - hãy xem liệu chúng ta có thể có được kết quả tương tự không:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Bạn có thể thấy rằng đám mây từ bây giờ trông ấn tượng hơn, nhưng nó cũng chứa nhiều nhiễu (ví dụ: các từ không liên quan như `Retrieved on`). Ngoài ra, chúng ta cũng có ít từ khóa gồm hai từ hơn, chẳng hạn như *data scientist* hoặc *computer science*. Điều này là do thuật toán RAKE làm tốt hơn trong việc chọn lựa từ khóa tốt từ văn bản. Ví dụ này minh họa tầm quan trọng của việc tiền xử lý và làm sạch dữ liệu, vì hình ảnh rõ ràng ở cuối sẽ cho phép chúng ta đưa ra quyết định tốt hơn.

Trong bài tập này, chúng ta đã trải qua một quy trình đơn giản để trích xuất một số ý nghĩa từ văn bản Wikipedia, dưới dạng từ khóa và đám mây từ. Ví dụ này khá đơn giản, nhưng nó thể hiện tốt tất cả các bước điển hình mà một nhà khoa học dữ liệu sẽ thực hiện khi làm việc với dữ liệu, bắt đầu từ thu thập dữ liệu cho đến trực quan hóa.

Trong khóa học của chúng ta, chúng ta sẽ thảo luận chi tiết về tất cả các bước đó.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
